In [56]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error, mean_absolute_percentage_error
from sklearn.decomposition import PCA
from sklearn.model_selection import GroupKFold
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV

## With VAC

In [57]:
df = pd.read_csv("../data/result_with_vac.csv")
df.describe()

,x,y,B,dB_dx,dB_dy,Jc,Jx,Jy,Ic,Rn,n
count,44608.00000,44608.000000,44608.000000,44608.000000,44608.000000,4.460800e+04,4.460800e+04,4.460800e+04,19571.000000,2.578700e+04,25787.000000
mean,4.55082,65.004131,0.000994,0.004992,0.020861,3.306073e+09,5.702838e+06,-8.273462e+07,130.071420,1.594162e-08,0.572158
std,3.03140,43.491752,0.005102,0.844270,3.490399,4.210129e+09,1.101466e+09,5.237885e+09,66.981945,1.367623e-08,0.911122
min,-1.44000,-20.010000,-0.007763,-12.134375,-11.983000,1.391623e+06,-2.192591e+10,-3.447863e+10,23.054000,-1.477728e-10,0.035860
25%,1.76000,30.000000,-0.002031,-0.053750,-0.839000,2.798265e+08,-7.462225e+07,-1.155763e+09,42.978000,3.463503e-10,0.055079
50%,4.73000,61.980000,-0.000668,0.000625,0.026250,1.139670e+09,2.913048e+06,-4.780780e+07,173.343300,1.092705e-08,0.158940
75%,7.17000,92.990000,0.001280,0.057500,0.870000,5.421462e+09,8.435189e+07,9.368305e+08,183.182300,2.952706e-08,0.448222
max,10.40000,226.900000,0.023210,13.345000,14.786000,3.468354e+10,1.779246e+10,2.704091e+10,187.745000,3.858297e-08,3.187189


In [58]:
df['log_Jc'] = np.log10(df['Jc'].replace(0, 1))
features = ['x', 'y', 'B', 'dB_dx', 'dB_dy', 'Ic', 'Rn', 'n']
target = 'log_Jc'

In [59]:
unique_titles = df['title'].unique()

train_val_titles, test_titles = train_test_split(
    unique_titles, test_size=0.15, random_state=42
)

train_titles, val_titles = train_test_split(
    train_val_titles, test_size=0.1765, random_state=42
)

In [60]:
df_train = df[df['title'].isin(train_titles)].copy()
df_val = df[df['title'].isin(val_titles)].copy()
df_test = df[df['title'].isin(test_titles)].copy()

In [61]:
print(f"Train: {df_train['title'].nunique()} samples, {len(df_train)} rows")
print(f"Validation: {df_val['title'].nunique()} samples, {len(df_val)} rows")
print(f"Test: {df_test['title'].nunique()} samples, {len(df_test)} rows")

Train: 22 samples, 30632 rows
Validation: 5 samples, 6779 rows
Test: 5 samples, 7197 rows


In [62]:
means = df_train[['Ic', 'Rn', 'n']].mean()

df_train[['Ic', 'Rn', 'n']] = df_train[['Ic', 'Rn', 'n']].fillna(means)
df_val[['Ic', 'Rn', 'n']] = df_val[['Ic', 'Rn', 'n']].fillna(means)
df_test[['Ic', 'Rn', 'n']] = df_test[['Ic', 'Rn', 'n']].fillna(means)
df_train_val = pd.concat([df_train, df_val])

In [63]:
X = df_train_val[features].values
y = df_train_val[target].values
groups = df_train_val['title'].values

In [64]:
X_test = df_test[features].values
y_test = df_test[target].values
X_val = df_val[features].values
y_val = df_val[target].values

In [65]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('poly', PolynomialFeatures(include_bias=False)),
    ('pca', PCA()),
    ('ridge', Ridge())
])

param_grid = {
    'poly__degree': [1, 2, 3, 4],
    'ridge__alpha': [0.01, 0.1, 1, 10, 100, 500],
}

In [66]:
gkf = GroupKFold(n_splits=5)

grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=gkf.split(X, y, groups=groups),
    scoring='neg_mean_absolute_error',
)

In [67]:
grid.fit(X, y)

GridSearchCV(cv=<generator object _BaseKFold.split at 0x0000018FCA9AF120>,
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('poly',
                                        PolynomialFeatures(include_bias=False)),
                                       ('pca', PCA()), ('ridge', Ridge())]),
             param_grid={'poly__degree': [1, 2, 3, 4],
                         'ridge__alpha': [0.01, 0.1, 1, 10, 100, 500]},
             scoring='neg_mean_absolute_error')

In [68]:
best_model = grid.best_estimator_
print("Best params:", grid.best_params_)

Best params: {'poly__degree': 3, 'ridge__alpha': 500}


In [69]:
def evaluate(y_true, y_pred, name):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = root_mean_squared_error(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    print(f"{name} R²: {r2:.4f}")
    print(f"{name} MAPE: {mape:.4f}")
    print(f"{name} MAE: {mae:.4f}")
    print(f"{name} RMSE: {rmse:.4f}")
    print()

In [70]:
evaluate(y_val, best_model.predict(X_val), "Validation")
evaluate(y_test, best_model.predict(X_test), "Test")

Validation R²: 0.7144
Validation MAPE: 0.0419
Validation MAE: 0.3542
Validation RMSE: 0.4415

Test R²: 0.7502
Test MAPE: 0.0347
Test MAE: 0.3001
Test RMSE: 0.3942



## Without VAC

In [20]:
df = pd.read_csv("../data/result.csv")
df.head()

,title,x,y,B,dB_dx,dB_dy,Jc,Jx,Jy
0,1,-0.01,0.0,-0.000325,-0.12875,-0.0760,1.090335e+08,7.915898e+07,7.498113e+07
1,1,0.79,0.0,-0.000401,-0.22875,-0.0965,9.263875e+07,8.640185e+06,9.223495e+07
2,1,1.59,0.0,-0.000518,-0.36750,-0.1235,4.150431e+07,3.640487e+07,1.993222e+07
3,1,2.39,0.0,-0.000648,-0.49875,-0.1220,4.602157e+08,-4.334496e+08,-1.546606e+08
4,1,3.19,0.0,-0.000762,-0.70125,-0.0860,1.537920e+09,-1.537311e+09,-4.328145e+07


In [27]:
df['log_Jc'] = np.log10(df['Jc'].replace(0, 1))
features = ['x', 'y', 'B', 'dB_dx', 'dB_dy']
target = 'log_Jc'

In [28]:
unique_titles = df['title'].unique()

train_val_titles, test_titles = train_test_split(
    unique_titles, test_size=0.15, random_state=42
)

train_titles, val_titles = train_test_split(
    train_val_titles, test_size=0.1765, random_state=42
)

In [29]:
df_train = df[df['title'].isin(train_titles)].copy()
df_val = df[df['title'].isin(val_titles)].copy()
df_test = df[df['title'].isin(test_titles)].copy()

In [30]:
print(f"Train: {df_train['title'].nunique()} samples, {len(df_train)} rows")
print(f"Validation: {df_val['title'].nunique()} samples, {len(df_val)} rows")
print(f"Test: {df_test['title'].nunique()} samples, {len(df_test)} rows")

Train: 22 samples, 30632 rows
Validation: 5 samples, 6779 rows
Test: 5 samples, 7197 rows


In [31]:
X = df_train_val[features].values
y = df_train_val[target].values
groups = df_train_val['title'].values

In [32]:
X_test = df_test[features].values
y_test = df_test[target].values
X_val = df_val[features].values
y_val = df_val[target].values

In [38]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('poly', PolynomialFeatures(include_bias=False)),
    ('pca', PCA()),
    ('ridge', Ridge())
])

param_grid = {
    'poly__degree': [1, 2, 3, 4, 5],
    'ridge__alpha': [0.01, 0.1, 1, 10, 100, 500],
}

In [39]:
gkf = GroupKFold(n_splits=5)

grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=gkf.split(X, y, groups=groups),
    scoring='neg_mean_absolute_error',
)

In [40]:
grid.fit(X, y)

GridSearchCV(cv=<generator object _BaseKFold.split at 0x0000018FCA9AE790>,
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('poly',
                                        PolynomialFeatures(include_bias=False)),
                                       ('pca', PCA()), ('ridge', Ridge())]),
             param_grid={'poly__degree': [1, 2, 3, 4, 5],
                         'ridge__alpha': [0.01, 0.1, 1, 10, 100, 500]},
             scoring='neg_mean_absolute_error')

In [41]:
best_model = grid.best_estimator_
print("Best params:", grid.best_params_)

Best params: {'poly__degree': 4, 'ridge__alpha': 100}


In [42]:
evaluate(y_val, best_model.predict(X_val), "Validation")
evaluate(y_test, best_model.predict(X_test), "Test")

Validation R²: 0.7301
Validation MAPE: 0.0398
Validation MAE: 0.3313
Validation RMSE: 0.4292

Test R²: 0.7804
Test MAPE: 0.0306
Test MAE: 0.2620
Test RMSE: 0.3697

